In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import img_to_array
from tensorflow.keras.optimizers import Adam

IMG_SIZE = 128

# CLASS LABELS
categories = ["fake", "genuine", "non_qr"]  # 0, 1, 2

DATASET_DIR = "dataset"

images = []
labels = []

print("Loading dataset...")

for idx, category in enumerate(categories):
    folder = os.path.join(DATASET_DIR, category)

    if not os.path.exists(folder):
        print("Folder missing:", folder)
        continue

    for file in os.listdir(folder):
        path = os.path.join(folder, file)

        if not file.lower().endswith((".jpg", ".jpeg", ".png")):
            continue

        img = cv2.imread(path)
        if img is None:
            continue

        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        img = img_to_array(img) / 255.0

        images.append(img)
        labels.append(idx)

images = np.array(images)
labels = to_categorical(np.array(labels), num_classes=3)

print(f"Total images loaded: {len(images)}")

# SPLIT TRAIN + TEST
X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.20, random_state=42
)

# MODEL
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(2, 2),

    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D(2, 2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')  # 3 classes
])

model.compile(
    optimizer=Adam(0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=16,
    validation_split=0.1
)

loss, acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {acc * 100:.2f}%")

model.save("model/qr_authenticator_model.h5")
print("Model saved successfully!")
